# Lightweight Multi-Task Brain MRI CNN
All modules assembled into a single notebook.
**Do not delete the original .py files.**

In [1]:
# Install dependencies
!pip install -q torch torchvision timm albumentations opencv-python scikit-learn seaborn onnx onnxruntime grad-cam tensorboard


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install --upgrade timm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Class balance Data

In [5]:
import os
import shutil
import random
import math

base_dir = r"../brisc-deepfakes/FullDataset"
output_dir = r"../brisc-deepfakes/ReadyDataset"


# Define sources, target destinations, and their desired fractions of the total dataset
# Note: real classes use 'no_tumor' in the workspace but we can rename it to 'notumor' in the target as requested
sources = [
    # 1. real Data (80% total -> 20% each)
    ("real/glioma", "real/glioma", 0.20),
    ("real/no_tumor", "real/notumor", 0.20),
    ("real/meningioma", "real/meningioma", 0.20),
    ("real/pituitary", "real/pituitary", 0.20),
    
    # 2. fake Data (20%) 
    # 2.1 gan-generated data (10% total -> 2.5% each)
    ("fake/gan/glioma", "fake/gan/glioma", 0.025),
    ("fake/gan/notumor", "fake/gan/notumor", 0.025),
    ("fake/gan/meningioma", "fake/gan/meningioma", 0.025),
    ("fake/gan/pituitary", "fake/gan/pituitary", 0.025),
    
    # 2.2 attacks Attacks (10% total -> 5% PGD, 5% FGSM)
    # PGD (5% total -> 1.25% each)
    ("fake/attacks/pgd/glioma", "fake/attacks/pgd/glioma", 0.0125),
    ("fake/attacks/pgd/notumor", "fake/attacks/pgd/notumor", 0.0125),
    ("fake/attacks/pgd/meningioma", "fake/attacks/pgd/meningioma", 0.0125),
    ("fake/attacks/pgd/pituitary", "fake/attacks/pgd/pituitary", 0.0125),
    
    # FGSM (5% total -> 1.25% each)
    ("fake/attacks/fgsm/glioma", "fake/attacks/fgsm/glioma", 0.0125),
    ("fake/attacks/fgsm/notumor", "fake/attacks/fgsm/notumor", 0.0125),
    ("fake/attacks/fgsm/meningioma", "fake/attacks/fgsm/meningioma", 0.0125),
    ("fake/attacks/fgsm/pituitary", "fake/attacks/fgsm/pituitary", 0.0125),
]

# Calculate the maximum possible dataset size based on the most limiting folder
max_n = float('inf')
counts_info = {}
limiting_class = ""

for src_rel, dest_rel, fraction in sources:
    src_path = os.path.join(base_dir, os.path.normpath(src_rel))
    if os.path.exists(src_path):
        # Count files
        files = [f for f in os.listdir(src_path) if os.path.isfile(os.path.join(src_path, f))]
        count = len(files)
        counts_info[src_rel] = {'count': count, 'files': files}
        
        # Calculate maximum possible total dataset size if this directory is the bottleneck
        possible_total = count / fraction
        if possible_total < max_n:
            max_n = possible_total
            limiting_class = src_rel
    else:
        print(f"Warning: source path {src_path} not found.")

max_n = math.floor(max_n)
print(f"Limiting class is '{limiting_class}' which allows a maximum total dataset size of {max_n} images.\n")

if max_n > 0:
    total_images_to_extract = max_n
    
    print("Started copying and balancing dataset:")
    # Process copying
    for src_rel, dest_rel, fraction in sources:
        target_count = int(total_images_to_extract * fraction)
        files = counts_info[src_rel]['files']
        
        # Sample randomly
        target_count = min(target_count, len(files))
        sampled_files = random.sample(files, target_count)
        
        dest_path = os.path.join(output_dir, os.path.normpath(dest_rel))
        os.makedirs(dest_path, exist_ok=True)
        
        for file in sampled_files:
            src_file = os.path.join(base_dir, os.path.normpath(src_rel), file)
            dest_file = os.path.join(dest_path, file)
            shutil.copy(src_file, dest_file)
            
        print(f"  -> Copied {target_count} images to {dest_rel} ({fraction*100}%)")
        
    print(f"\n✅ Ready-Dataset compilation successfully finished at: {output_dir}")
else:
    print("Error: Could not determine total images due to missing files or empty directories.")

Limiting class is 'real/no_tumor' which allows a maximum total dataset size of 6035 images.

Started copying and balancing dataset:
  -> Copied 1207 images to real/glioma (20.0%)
  -> Copied 1207 images to real/notumor (20.0%)
  -> Copied 1207 images to real/meningioma (20.0%)
  -> Copied 1207 images to real/pituitary (20.0%)
  -> Copied 150 images to fake/gan/glioma (2.5%)
  -> Copied 150 images to fake/gan/notumor (2.5%)
  -> Copied 150 images to fake/gan/meningioma (2.5%)
  -> Copied 150 images to fake/gan/pituitary (2.5%)
  -> Copied 75 images to fake/attacks/pgd/glioma (1.25%)
  -> Copied 75 images to fake/attacks/pgd/notumor (1.25%)
  -> Copied 75 images to fake/attacks/pgd/meningioma (1.25%)
  -> Copied 75 images to fake/attacks/pgd/pituitary (1.25%)
  -> Copied 75 images to fake/attacks/fgsm/glioma (1.25%)
  -> Copied 75 images to fake/attacks/fgsm/notumor (1.25%)
  -> Copied 75 images to fake/attacks/fgsm/meningioma (1.25%)
  -> Copied 75 images to fake/attacks/fgsm/pituitary 

In [25]:
!ls /kaggle/working/

checkpoints  export  Ready-Dataset  results  runs


---
## config.py

In [22]:
"""
config.py
---------
Central configuration for the Lightweight Multi-Task Brain MRI CNN.
All hyper-parameters, paths and class definitions are defined here.
"""

import os
from dataclasses import dataclass, field
from typing import List, Optional


# ─────────────────────────────────────────────────────────────────────────────
# Class Definitions
# ─────────────────────────────────────────────────────────────────────────────

# Authenticity labels  (head 1)
AUTHENTICITY_CLASSES = ["real", "fake"]          # binary
AUTHENTICITY_IDX     = {c: i for i, c in enumerate(AUTHENTICITY_CLASSES)}

# Tumor-type labels   (head 2, only for real images)
TUMOR_CLASSES  = ["no_tumor", "glioma", "meningioma", "pituitary"]
TUMOR_IDX      = {c: i for i, c in enumerate(TUMOR_CLASSES)}
NUM_TUMOR_CLASSES = len(TUMOR_CLASSES)


# ─────────────────────────────────────────────────────────────────────────────
# 5-way flat label space used for the joint confusion matrix
# ─────────────────────────────────────────────────────────────────────────────
JOINT_CLASSES = ["no_tumor", "glioma", "meningioma", "pituitary", "fake"]
NUM_JOINT_CLASSES = len(JOINT_CLASSES)  # 5


@dataclass
class DataConfig:
    """Dataset and augmentation settings."""
    # ── root structure expected ──────────────────────────────────────────────
    # data_root/
    #   real/
    #     notumor/  glioma/  meningioma/  pituitary/
    #   fake/
    #     gan/      attacks/
    # ────────────────────────────────────────────────────────────────────────
    data_root: str        = "../brisc-deepfakes/ReadyDataset"
    image_size: int       = 224          # MobileViT / ShuffleNet input
    mean: List[float]     = field(default_factory=lambda: [0.485, 0.456, 0.406])
    std:  List[float]     = field(default_factory=lambda: [0.229, 0.224, 0.225])

    # ── augmentation strengths ───────────────────────────────────────────────
    aug_rotation_limit:  int   = 20
    aug_brightness:      float = 0.2
    aug_contrast:        float = 0.2
    aug_noise_var_limit: tuple = (10.0, 50.0)
    aug_elastic_alpha:   int   = 120
    aug_elastic_sigma:   int   = 6

    # ── splitting ────────────────────────────────────────────────────────────
    val_split:  float = 0.10
    test_split: float = 0.20
    k_folds:    int   = 5        # for k-fold CV
    random_seed: int  = 42


@dataclass
class ModelConfig:
    """Architecture settings."""
    backbone: str = "lcnet_050"
    pretrained: bool = True
    freeze_backbone_epochs: int = 5    # freeze during stage-1 warm-up
    dropout_rate: float     = 0.5
    embedding_dim: int      = 512       # shared feature dim after backbone

    # ── head dimensions ──────────────────────────────────────────────────────
    auth_head_hidden:      int = 256    # real vs fake head
    tumor_head_hidden:     int = 256    # Tumor-type head


@dataclass
class TrainingConfig:
    """Training and optimiser settings."""
    # ── stage-1: authenticity pre-training ──────────────────────────────────
    stage1_epochs: int   = 20
    stage1_lr:     float = 1e-3
    stage1_batch:  int   = 32

    # ── stage-2: tumor head only (backbone + auth head frozen) ───────────────
    stage2_epochs: int   = 15
    stage2_lr:     float = 5e-4
    stage2_batch:  int   = 32

    # ── stage-3: joint fine-tuning ───────────────────────────────────────────
    stage3_epochs: int   = 30
    stage3_lr:     float = 1e-4
    stage3_batch:  int   = 16

    # ── loss weights (stage 3 joint) ─────────────────────────────────────────
    loss_weight_auth:      float = 1.0
    loss_weight_tumor:     float = 1.5   # up-weight clinical task

    # ── optimiser ────────────────────────────────────────────────────────────
    optimizer:     str   = "adam"        # "adam" | "sgd"
    weight_decay:  float = 1e-3
    scheduler:     str   = "cosine"      # "cosine" | "step"
    warmup_epochs: int   = 3

    # ── misc ─────────────────────────────────────────────────────────────────
    num_workers:   int   = 4
    pin_memory:    bool  = True
    mixed_precision: bool = True         # AMP fp16
    grad_clip:     float = 1.0
    early_stop_patience: int = 10


@dataclass
class EvalConfig:
    """Evaluation and export settings."""
    metrics: List[str] = field(default_factory=lambda: [
        "accuracy", "f1_macro", "f1_per_class", "auc_roc", "confusion_matrix"
    ])
    grad_cam_layer: str = "backbone.conv5"   # layer for Grad-CAM visualisation

    # ── ONNX export ──────────────────────────────────────────────────────────
    export_onnx:        bool = True
    onnx_opset:         int  = 17
    quantize_int8:      bool = True          # post-training quantization


@dataclass
class PathConfig:
    """Output path settings."""
    checkpoint_dir: str = "checkpoints"
    log_dir:        str = "runs"
    onnx_dir:       str = "export"
    results_dir:    str = "results"

    def make_dirs(self):
        for d in [self.checkpoint_dir, self.log_dir,
                  self.onnx_dir, self.results_dir]:
            os.makedirs(d, exist_ok=True)


@dataclass
class Config:
    """Master config – compose all sub-configs."""
    data:     DataConfig     = field(default_factory=DataConfig)
    model:    ModelConfig    = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    eval:     EvalConfig     = field(default_factory=EvalConfig)
    paths:    PathConfig     = field(default_factory=PathConfig)
    device:   str            = "cuda"   # "cuda" | "cpu"
    amp:      bool           = True     # automatic mixed precision
    seed:     int            = 42


# Singleton used across the project
CFG = Config()


---
## utils.py

In [23]:
"""
utils.py
--------
Shared utilities:
  - set_seed
  - count_parameters
  - EarlyStopping class
  - LabelSmoothingCrossEntropy
  - Mixup augmentation
  - AverageMeter
  - format_time
"""

from __future__ import annotations

import math
import random
import time
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ─────────────────────────────────────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────────────────────────────────────
# Parameter counting
# ─────────────────────────────────────────────────────────────────────────────

def count_parameters(model: nn.Module) -> Tuple[int, int]:
    """Returns (total_params, trainable_params)."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ─────────────────────────────────────────────────────────────────────────────
# Early Stopping
# ─────────────────────────────────────────────────────────────────────────────

class EarlyStopping:
    """
    Stop training when monitored metric stops improving.

    Parameters
    ----------
    patience  : epochs to wait after last improvement
    min_delta : minimum change to qualify as improvement
    mode      : 'min' (loss) or 'max' (accuracy)
    """

    def __init__(self, patience: int = 10, min_delta: float = 1e-4,
                 mode: str = "min"):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.best      = float("inf") if mode == "min" else -float("inf")
        self.counter   = 0
        self.triggered = False

    def __call__(self, metric: float) -> bool:
        if self.mode == "min":
            improved = metric < self.best - self.min_delta
        else:
            improved = metric > self.best + self.min_delta

        if improved:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


# ─────────────────────────────────────────────────────────────────────────────
# Label Smoothing Cross-Entropy
# ─────────────────────────────────────────────────────────────────────────────

class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-entropy with label smoothing. Handles ignore_index=-1.

    Parameters
    ----------
    smoothing    : float in [0, 1), 0 = standard CE
    ignore_index : label to ignore (e.g. masked samples)
    """

    def __init__(self, smoothing: float = 0.1, ignore_index: int = -1):
        super().__init__()
        self.smoothing     = smoothing
        self.ignore_index  = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        mask = (targets != self.ignore_index)
        if not mask.any():
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        logits  = logits[mask]
        targets = targets[mask]

        n_classes = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)

        # Smooth target distribution
        with torch.no_grad():
            smooth_targets = torch.full_like(log_probs,
                                             self.smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.smoothing)

        return -(smooth_targets * log_probs).sum(dim=-1).mean()


# ─────────────────────────────────────────────────────────────────────────────
# Mixup augmentation (image-level; applied only to real samples)
# ─────────────────────────────────────────────────────────────────────────────

def mixup_data(x: torch.Tensor, y: torch.Tensor,
               alpha: float = 0.2) -> Tuple[torch.Tensor, torch.Tensor,
                                            torch.Tensor, float]:
    """Apply Mixup to a batch (x, y). Returns (mixed_x, y_a, y_b, lam)."""
    if alpha > 0:
        lam = float(np.random.beta(alpha, alpha))
    else:
        lam = 1.0

    batch_size = x.size(0)
    idx        = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute Mixup loss."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ─────────────────────────────────────────────────────────────────────────────
# AverageMeter
# ─────────────────────────────────────────────────────────────────────────────

class AverageMeter:
    """Tracks running average of a scalar metric."""

    def __init__(self, name: str = ""):
        self.name = name
        self.reset()

    def reset(self):
        self.val   = 0.0
        self.avg   = 0.0
        self.sum   = 0.0
        self.count = 0

    def update(self, val: float, n: int = 1):
        self.val    = val
        self.sum   += val * n
        self.count += n
        self.avg    = self.sum / self.count

    def __str__(self):
        return f"{self.name}: {self.avg:.4f}"


# ─────────────────────────────────────────────────────────────────────────────
# Time formatting
# ─────────────────────────────────────────────────────────────────────────────

def format_time(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


# ─────────────────────────────────────────────────────────────────────────────
# Warmup LR scheduler wrapper
# ─────────────────────────────────────────────────────────────────────────────

class WarmupScheduler:
    """
    Linear warmup for the first `warmup_steps` steps,
    then delegates to `base_scheduler`.
    """

    def __init__(self, optimizer, warmup_steps: int, base_scheduler):
        self.optimizer      = optimizer
        self.warmup_steps   = warmup_steps
        self.base_scheduler = base_scheduler
        self._step          = 0
        self._base_lrs      = [pg["lr"] for pg in optimizer.param_groups]
        # Set LR to near zero
        for pg in optimizer.param_groups:
            pg["lr"] = 1e-8

    def step(self):
        self._step += 1
        if self._step <= self.warmup_steps:
            frac = self._step / self.warmup_steps
            for pg, base_lr in zip(self.optimizer.param_groups, self._base_lrs):
                pg["lr"] = base_lr * frac
        else:
            self.base_scheduler.step()


---
## dataset.py

In [24]:
"""
dataset.py
----------
MRI multi-task dataset with:
  - MRI-specific augmentation (intensity normalization, noise, elastic)
  - Balanced sampling via WeightedRandomSampler
  - Patient-level train/val/test splitting (no data leakage)
  - Returns (image, auth_label, tumor_label, joint_label)
    where tumor_label = -1 for fake images (masked in loss)
"""

import os
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# from config import (CFG, AUTHENTICITY_IDX, TUMOR_IDX, FAKE_TYPE_IDX,
#                     JOINT_CLASSES, NUM_JOINT_CLASSES)


# ─────────────────────────────────────────────────────────────────────────────
# Augmentation pipelines
# ─────────────────────────────────────────────────────────────────────────────

def build_train_transforms(cfg=CFG) -> A.Compose:
    """Heavy MRI-specific augmentation for training."""
    dc = cfg.data
    return A.Compose([
        A.Resize(dc.image_size, dc.image_size),

        # ── Spatial ──────────────────────────────────────────────────────────
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Rotate(limit=dc.aug_rotation_limit, p=0.7,
                 border_mode=cv2.BORDER_REFLECT_101),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=0, p=0.4),
        A.ElasticTransform(alpha=dc.aug_elastic_alpha,
                           sigma=dc.aug_elastic_sigma,
                           p=0.3),

        # ── Intensity / MRI-specific ─────────────────────────────────────────
        A.RandomBrightnessContrast(brightness_limit=dc.aug_brightness,
                                   contrast_limit=dc.aug_contrast, p=0.6),
        A.GaussNoise(var_limit=dc.aug_noise_var_limit, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),

        # ── Occlusion ────────────────────────────────────────────────────────
        A.CoarseDropout(max_holes=8, max_height=16, max_width=16,
                        fill_value=0, p=0.2),

        # ── Normalize & tensor ───────────────────────────────────────────────
        A.Normalize(mean=dc.mean, std=dc.std),
        ToTensorV2(),
    ])


def build_val_transforms(cfg=CFG) -> A.Compose:
    """Deterministic pipeline for validation / test."""
    dc = cfg.data
    return A.Compose([
        A.Resize(dc.image_size, dc.image_size),
        A.Normalize(mean=dc.mean, std=dc.std),
        ToTensorV2(),
    ])


# ─────────────────────────────────────────────────────────────────────────────
# Sample catalogue builder
# ─────────────────────────────────────────────────────────────────────────────

def _joint_label(auth: int, tumor: int) -> int:
    if auth == AUTHENTICITY_IDX["real"]:
        return tumor  # 0..3
    else:
        return 4  # fake (single class)



def scan_dataset(root: str) -> List[Dict]:
    """
    Walk the directory tree and build a flat list of sample dicts.

    Expected layout::

        <root>/
          real/
            notumor/    *.jpg | *.png | *.jpeg
            glioma/
            meningioma/
            pituitary/
          fake/
            gan/
            attacks/
    """
    samples: List[Dict] = []
    root = Path(root)

    for subdir in root.rglob("*"):
        if not subdir.is_file():
            continue
        if subdir.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
            continue

        parts = [p.lower() for p in subdir.parts]

        # ── Determine authenticity ────────────────────────────────────────────
        if "real" in parts:
            auth = AUTHENTICITY_IDX["real"]
            # Tumor type from parent folder name
            folder = subdir.parent.name.lower()
            tumor_map = {
                "notumor": TUMOR_IDX["no_tumor"],
                "no_tumor": TUMOR_IDX["no_tumor"],
                "glioma": TUMOR_IDX["glioma"],
                "meningioma": TUMOR_IDX["meningioma"],
                "pituitary": TUMOR_IDX["pituitary"],
            }
            tumor_label = tumor_map.get(folder, -1)
            if tumor_label == -1:
                continue   # unknown sub-folder, skip

        elif "fake" in parts:
            auth = AUTHENTICITY_IDX["fake"]
            tumor_label = -1  # masked
        else:
            continue  # unknown top-level dir

        joint = _joint_label(auth, tumor_label)

        samples.append({
            "path":  str(subdir),
            "auth":  auth,
            "tumor": tumor_label,
            "joint": joint,
        })

    return samples


# ─────────────────────────────────────────────────────────────────────────────
# Patient-level stratified split (prevents data leakage)
# ─────────────────────────────────────────────────────────────────────────────

def stratified_patient_split(
    samples: List[Dict],
    val_frac: float = 0.15,
    test_frac: float = 0.10,
    seed: int = 42,
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """
    Split samples into train / val / test.

    Here we use the directory as a proxy for "patient group" — in a real
    scenario you'd group by patient ID embedded in the filename.
    """
    rng = random.Random(seed)

    # Group by joint label to keep stratification
    from collections import defaultdict
    buckets: Dict[int, List[Dict]] = defaultdict(list)
    for s in samples:
        buckets[s["joint"]].append(s)

    train_list, val_list, test_list = [], [], []

    for label, group in buckets.items():
        rng.shuffle(group)
        n = len(group)
        n_test = max(1, int(n * test_frac))
        n_val  = max(1, int(n * val_frac))
        test_list  += group[:n_test]
        val_list   += group[n_test:n_test + n_val]
        train_list += group[n_test + n_val:]

    return train_list, val_list, test_list


# ─────────────────────────────────────────────────────────────────────────────
# PyTorch Dataset
# ─────────────────────────────────────────────────────────────────────────────

class MRIMultiTaskDataset(Dataset):
    """
    Returns
    -------
    image          : FloatTensor (C, H, W)
    auth_label     : LongTensor scalar   – 0=real, 1=fake
    tumor_label    : LongTensor scalar   – 0-3 or -1 (masked)
    joint_label    : LongTensor scalar   – 0-4 (5-way)
    """

    def __init__(self, samples: List[Dict], transform: A.Compose):
        self.samples   = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        s = self.samples[idx]

        # ── Load & convert to RGB ─────────────────────────────────────────────
        img = cv2.imread(s["path"], cv2.IMREAD_COLOR)
        if img is None:
            # Fallback: black image
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # ── Intensity normalization (MRI-specific: per-image z-score) ─────────
        img = img.astype(np.float32)
        mean_i, std_i = img.mean(), img.std() + 1e-6
        img = ((img - mean_i) / std_i * 50 + 128).clip(0, 255).astype(np.uint8)

        augmented = self.transform(image=img)
        image_tensor = augmented["image"]   # (C, H, W) float32

        return (
            image_tensor,
            torch.tensor(s["auth"],  dtype=torch.long),
            torch.tensor(s["tumor"], dtype=torch.long),
            torch.tensor(s["joint"], dtype=torch.long),
        )


# ─────────────────────────────────────────────────────────────────────────────
# Balanced sampler (handles class imbalance)
# ─────────────────────────────────────────────────────────────────────────────

def make_weighted_sampler(samples: List[Dict]) -> WeightedRandomSampler:
    """Inverse-frequency weighting over joint 5-way labels."""
    from collections import Counter

    counts  = Counter(s["joint"] for s in samples)
    total   = sum(counts.values())
    weights = [total / counts[s["joint"]] for s in samples]

    return WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True,
    )


# ─────────────────────────────────────────────────────────────────────────────
# DataLoader factory
# ─────────────────────────────────────────────────────────────────────────────

def build_dataloaders(cfg=CFG, fold_idx: Optional[int] = None):
    """
    Build train / val / test DataLoaders.

    Parameters
    ----------
    cfg      : Config
    fold_idx : if not None, use k-fold; else use holdout split.

    Returns
    -------
    train_loader, val_loader, test_loader
    """
    all_samples = scan_dataset(cfg.data.data_root)
    if not all_samples:
        raise FileNotFoundError(
            f"No samples found under '{cfg.data.data_root}'. "
            "Check directory structure: real/{{notumor,glioma,…}} "
            "and fake/{{gan,attacks}}."
        )

    train_s, val_s, test_s = stratified_patient_split(
        all_samples,
        val_frac=cfg.data.val_split,
        test_frac=cfg.data.test_split,
        seed=cfg.data.random_seed,
    )

    train_tf = build_train_transforms(cfg)
    val_tf   = build_val_transforms(cfg)

    train_ds = MRIMultiTaskDataset(train_s, train_tf)
    val_ds   = MRIMultiTaskDataset(val_s,   val_tf)
    test_ds  = MRIMultiTaskDataset(test_s,  val_tf)

    sampler = make_weighted_sampler(train_s)

    tc = cfg.training
    train_loader = DataLoader(
        train_ds, batch_size=tc.stage1_batch, sampler=sampler,
        num_workers=tc.num_workers, pin_memory=tc.pin_memory,
        drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=tc.stage1_batch * 2, shuffle=False,
        num_workers=tc.num_workers, pin_memory=tc.pin_memory,
    )
    test_loader = DataLoader(
        test_ds, batch_size=tc.stage1_batch * 2, shuffle=False,
        num_workers=tc.num_workers, pin_memory=tc.pin_memory,
    )

    print(f"[Dataset] train={len(train_ds):,}  val={len(val_ds):,}  "
          f"test={len(test_ds):,}  total={len(all_samples):,}")
    return train_loader, val_loader, test_loader


---
## model.py

In [25]:
"""
model.py
--------
Lightweight multi-task CNN for brain MRI.

Architecture
────────────
                    ┌─────────────────────────┐
                    │   ShuffleNetV2-1.0x      │  ← ImageNet pretrained
                    │   (backbone)             │
                    └──────────┬──────────────┘
                               │  feature vector (1024-d)
                    ┌──────────▼──────────────┐
                    │  Shared Embedding FC     │  (→ 512-d, BN, ReLU, Drop)
                    └──┬───────────┬──────────┘
                       │           │
            ┌──────────▼──┐  ┌────▼──────────┐
            │ Auth Head   │  │ Backbone feats │
            │ real / fake │  │ (passed thru)  │
            └──────────┬──┘  └────┬──────────┘
                       │           │
              ┌────────▼───┐  ┌───▼──────────────┐
              │ Tumor Head │  │  fake-Type Head   │
              │ (if real)  │  │  gan / Adv        │
              │  4-way     │  │  (if fake)        │
              └────────────┘  └──────────────────┘

Note: tumor head and fake-type head both receive the shared embedding.
      At inference, only the relevant head is activated based on auth output.
"""

from __future__ import annotations
from typing import Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# from config import (CFG, ModelConfig,
#                     NUM_TUMOR_CLASSES, NUM_FAKE_TYPES,
#                     AUTHENTICITY_IDX)


# ─────────────────────────────────────────────────────────────────────────────
# Building-block helpers
# ─────────────────────────────────────────────────────────────────────────────

class ClassificationHead(nn.Module):
    """Generic MLP classification head."""

    def __init__(self, in_dim: int, hidden_dim: int,
                 num_classes: int, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ─────────────────────────────────────────────────────────────────────────────
# Main multi-task model
# ─────────────────────────────────────────────────────────────────────────────

class MultiTaskMRINet(nn.Module):
    """
    Multi-task CNN with three output heads:
      - auth_logits      (2)   real vs fake
      - tumor_logits     (4)   Tumor type  (valid only for real)
    

    Parameters
    ----------
    cfg : ModelConfig
    """

    def __init__(self, cfg: ModelConfig = CFG.model):
        super().__init__()
        self.cfg = cfg

        # ── Backbone ─────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=cfg.pretrained,
            num_classes=0,        # remove classifier head
            global_pool="avg",    # global average pool → flat vector
        )
        
        # ✅ Dynamically infer backbone output dimension
        with torch.no_grad():
            _dummy = torch.zeros(1, 3, 224, 224)
            backbone_out_dim = self.backbone(_dummy).shape[1]
        
        # ── Shared embedding ─────────────────────────────────────────────────
        self.shared_embed = nn.Sequential(
            nn.Linear(backbone_out_dim, cfg.embedding_dim),  # ✅ dynamic
            nn.BatchNorm1d(cfg.embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.dropout_rate),
        )

        # ── Head 1: Authenticity (real / fake) ───────────────────────────────
        self.auth_head = ClassificationHead(
            in_dim=cfg.embedding_dim,
            hidden_dim=cfg.auth_head_hidden,
            num_classes=2,
            dropout=cfg.dropout_rate,
        )

        # ── Head 2: Tumor type (4-way, real images only) ─────────────────────
        self.tumor_head = ClassificationHead(
            in_dim=cfg.embedding_dim,
            hidden_dim=cfg.tumor_head_hidden,
            num_classes=NUM_TUMOR_CLASSES,
            dropout=cfg.dropout_rate,
        )

        self._init_heads()

    # ── Weight initialisation ─────────────────────────────────────────────────
    def _init_heads(self):
        for module in [self.shared_embed, self.auth_head, self.tumor_head]:
            for m in module.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, nn.BatchNorm1d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    # ── Forward ───────────────────────────────────────────────────────────────
    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns
        -------
        auth_logits  : (B, 2)
        tumor_logits : (B, 4)

        During training both are always returned; masking is handled in
        the loss function.  During inference, use `predict()` for cleaner API.
        """
        feats  = self.backbone(x)            # (B, backbone_out_dim)
        embed  = self.shared_embed(feats)    # (B, 512)

        auth_logits  = self.auth_head(embed)   # (B, 2)
        tumor_logits = self.tumor_head(embed)  # (B, 4)

        return auth_logits, tumor_logits

    # ── Inference helper ──────────────────────────────────────────────────────
    @torch.inference_mode()
    def predict(self, x: torch.Tensor) -> dict:
        """
        Runs full cascaded inference.

        Returns
        -------
        dict with keys:
          auth_pred    int   (0=real, 1=fake)
          auth_prob    float
          tumor_pred   int | None   (None if fake)
          tumor_probs  list[float] | None
          joint_label  int   (0-4)
        """
        auth_logits, tumor_logits = self.forward(x)

        auth_probs = F.softmax(auth_logits, dim=-1)
        auth_pred  = int(auth_probs.argmax(-1).item())
        auth_prob  = float(auth_probs.max(-1).values.item())

        if auth_pred == AUTHENTICITY_IDX["real"]:
            tumor_probs = F.softmax(tumor_logits, dim=-1)
            tumor_pred  = int(tumor_probs.argmax(-1).item())
            joint       = tumor_pred          # 0-3
            return {
                "auth_pred":      auth_pred,
                "auth_prob":      auth_prob,
                "tumor_pred":     tumor_pred,
                "tumor_probs":    tumor_probs[0].tolist(),
                "joint_label":    joint,
            }
        else:
            joint = 4  # fake
            return {
                "auth_pred":   auth_pred,
                "auth_prob":   auth_prob,
                "tumor_pred":  None,
                "tumor_probs": None,
                "joint_label": joint,
            }

    # ── Backbone freeze / unfreeze ────────────────────────────────────────────
    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(True)

    def freeze_auth_head(self):
        for p in self.auth_head.parameters():
            p.requires_grad_(False)

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad_(True)

    # ── Parameter count ──────────────────────────────────────────────────────
    def count_parameters(self) -> dict:
        total    = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable,
                "total_MB": total * 4 / 1e6}


# ─────────────────────────────────────────────────────────────────────────────
# Loss function
# ─────────────────────────────────────────────────────────────────────────────

class MultiTaskLoss(nn.Module):
    """
    Combines three cross-entropy losses with masking.

    fake samples contribute to auth loss only.
    real samples contribute to auth loss + tumor loss only.
    Label == -1 → that sample is ignored for that head.
    """

    def __init__(self,
                 w_auth:  float = 1.0,
                 w_tumor: float = 1.5):
        super().__init__()
        self.w_auth  = w_auth
        self.w_tumor = w_tumor

        # ignore_index=-1 means masked samples contribute zero loss
        self.ce = nn.CrossEntropyLoss(ignore_index=-1, reduction="mean", label_smoothing=0.1)

    def forward(self, auth_logits, tumor_logits,
            auth_labels, tumor_labels):

        device = auth_logits.device
    
        # Auth always has valid labels — never masked
        loss_auth = self.ce(auth_logits, auth_labels)
    
        # Slice to only valid (non-masked) real samples
        real_mask = (tumor_labels != -1)
        if self.w_tumor > 0 and real_mask.any():
            loss_tumor = self.ce(tumor_logits[real_mask], tumor_labels[real_mask])
        else:
            loss_tumor = torch.zeros(1, device=device).squeeze()
    
        total = (self.w_auth  * loss_auth
               + self.w_tumor * loss_tumor)
    
        return total, {
            "loss_auth":   loss_auth.item(),
            "loss_tumor":  loss_tumor.item(),
            "loss_total":  total.item(),
        }
# ─────────────────────────────────────────────────────────────────────────────
# Quick sanity check
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import time

    net = MultiTaskMRINet()
    net.eval()

    dummy = torch.randn(2, 3, 224, 224)
    t0    = time.time()
    a, t = net(dummy)
    elapsed = (time.time() - t0) * 1000

    stats = net.count_parameters()
    print(f"\n{'='*60}")
    print(f"  Model : {CFG.model.backbone}")
    print(f"  Params: {stats['total']:,}  ({stats['total_MB']:.1f} MB fp32)")
    print(f"  Auth  : {a.shape}   Tumor: {t.shape}")
    print(f"  Inference time (batch=2): {elapsed:.1f} ms")
    print(f"{'='*60}\n")

    # Loss sanity
    criterion = MultiTaskLoss()
    auth_gt  = torch.tensor([0, 1])  # real, fake
    tumor_gt = torch.tensor([1, -1])  # glioma, masked
    loss, details = criterion(a, t, auth_gt, tumor_gt)
    print("Loss components:", details)



  Model : lcnet_050
  Params: 1,521,974  (6.1 MB fp32)
  Auth  : torch.Size([2, 2])   Tumor: torch.Size([2, 4])
  Inference time (batch=2): 20.2 ms

Loss components: {'loss_auth': 0.7514728903770447, 'loss_tumor': 1.5058280229568481, 'loss_total': 3.0102150440216064}


---
## train.py

In [ ]:
"""
train.py
--------
Staged training pipeline for MultiTaskMRINet:

  Stage 1  – Authenticity head warm-up (real vs fake)
             Backbone frozen (ImageNet features preserved)
  Stage 2  – Tumor head only (backbone + auth head frozen)
             Trained on real samples only (fake loss is masked)
  Stage 3  – Joint fine-tuning (all heads, unfrozen backbone)
             Multi-task loss with weighted components

Usage
-----
    python train.py                        # defaults
    python train.py --data data/           # custom data root
    python train.py --stage 3              # run only stage 3

TensorBoard logs written to runs/
Checkpoints written to checkpoints/
"""

from __future__ import annotations

import argparse
import json
import os
import random
import time
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import os
# from config import CFG, Config, TrainingConfig
# from dataset import build_dataloaders
# from model import MultiTaskMRINet, MultiTaskLoss
# from evaluate import run_evaluation


# ─────────────────────────────────────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────────────────────────────────────

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────────────────────────────────────
# Optimiser + Scheduler factories
# ─────────────────────────────────────────────────────────────────────────────

def build_optimizer(model: MultiTaskMRINet, lr: float, tc: TrainingConfig):
    params = filter(lambda p: p.requires_grad, model.parameters())
    if tc.optimizer.lower() == "adam":
        return optim.AdamW(params, lr=lr, weight_decay=tc.weight_decay)
    return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=tc.weight_decay)


def build_scheduler(optimizer, tc: TrainingConfig, num_epochs: int, steps_per_epoch: int):
    if tc.scheduler == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs * steps_per_epoch, eta_min=1e-6
        )
    return optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


# ─────────────────────────────────────────────────────────────────────────────
# One-epoch training loop
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(
    model:     MultiTaskMRINet,
    loader,
    criterion: MultiTaskLoss,
    optimizer,
    scheduler,
    scaler:    GradScaler,
    device:    torch.device,
    cfg:       Config,
    epoch:     int,
    writer:    SummaryWriter,
    stage:     int,
) -> Dict[str, float]:

    model.train()
    tc   = cfg.training
    total_loss = 0.0
    comp_losses: Dict[str, float] = {
        "loss_auth": 0, "loss_tumor": 0
    }
    n_batches = 0

    pbar = tqdm(loader, desc=f"Stage {stage}  Epoch {epoch:03d}  Train",
                leave=False, ncols=90)

    for imgs, auth_gt, tumor_gt, _ in pbar:
        imgs     = imgs.to(device,     non_blocking=True)
        auth_gt  = auth_gt.to(device,  non_blocking=True)
        tumor_gt = tumor_gt.to(device, non_blocking=True)

    
        # ── MixUp augmentation (image only, labels unchanged) ────────────────
        if np.random.rand() < 0.3:
            lam = np.random.beta(0.4, 0.4)
            idx = torch.randperm(imgs.size(0), device=device)
            imgs = lam * imgs + (1 - lam) * imgs[idx]
    
        optimizer.zero_grad(set_to_none=True)
    
        with autocast("cuda", enabled=cfg.amp):
            a_logits, t_logits = model(imgs)
            loss, details = criterion(
                a_logits, t_logits,
                auth_gt, tumor_gt
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += details["loss_total"]
        for k in comp_losses:
            comp_losses[k] += details[k]
        n_batches += 1

        pbar.set_postfix(loss=f"{details['loss_total']:.4f}")

    avg = {k: v / max(n_batches, 1) for k, v in comp_losses.items()}
    avg["loss_total"] = total_loss / max(n_batches, 1)

    step = epoch * len(loader)
    for k, v in avg.items():
        writer.add_scalar(f"Stage{stage}/train/{k}", v, step)
    writer.add_scalar(f"Stage{stage}/train/lr",
                      optimizer.param_groups[0]["lr"], step)

    return avg


# ─────────────────────────────────────────────────────────────────────────────
# Validation loop
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def validate(
    model:     MultiTaskMRINet,
    loader,
    criterion: MultiTaskLoss,
    device:    torch.device,
    cfg:       Config,
    epoch:     int,
    writer:    SummaryWriter,
    stage:     int,
) -> float:
    model.eval()
    total_loss = 0.0
    n = 0

    for imgs, auth_gt, tumor_gt, _ in tqdm(
            loader, desc="  Val ", leave=False, ncols=90):
        imgs     = imgs.to(device)
        auth_gt  = auth_gt.to(device)
        tumor_gt = tumor_gt.to(device)
        with autocast("cuda", enabled=cfg.amp):
            a_logits, t_logits = model(imgs)
            loss, _ = criterion(a_logits, t_logits,
                                auth_gt, tumor_gt)
        total_loss += loss.item()
        n += 1

    avg_loss = total_loss / max(n, 1)
    writer.add_scalar(f"Stage{stage}/val/loss", avg_loss, epoch)
    return avg_loss


# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint helpers
# ─────────────────────────────────────────────────────────────────────────────

def save_checkpoint(model: MultiTaskMRINet, path: str, meta: dict):
    torch.save({"model_state": model.state_dict(), **meta}, path)
    print(f"  ✓ Checkpoint saved → {path}")


def load_checkpoint(model: MultiTaskMRINet, path: str, device: torch.device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    return ckpt


# ─────────────────────────────────────────────────────────────────────────────
# Stage runners
# ─────────────────────────────────────────────────────────────────────────────

def run_stage(
    stage:  int,
    model:  MultiTaskMRINet,
    loaders: tuple,
    cfg:    Config,
    device: torch.device,
    writer: SummaryWriter,
    start_epoch: int = 1,
) -> str:
    """Generic stage runner. Returns path to best checkpoint."""
    tc  = cfg.training
    train_loader, val_loader, _ = loaders

    # ── Stage-specific setup ─────────────────────────────────────────────────
    if stage == 1:
        epochs, lr = tc.stage1_epochs, tc.stage1_lr
        model.freeze_backbone()
        # Only auth head is active; tumor & fake-type losses will be masked
        criterion = MultiTaskLoss(w_auth=1.0, w_tumor=0.0)
        ckpt_name = "stage1_auth_best.pt"

    elif stage == 2:
        epochs, lr = tc.stage2_epochs, tc.stage2_lr
        model.freeze_backbone()
        model.freeze_auth_head()
        criterion = MultiTaskLoss(w_auth=0.3, w_tumor=1.0)
        ckpt_name = "stage2_tumor_best.pt"

    else:  # stage == 3
        epochs, lr = tc.stage3_epochs, tc.stage3_lr
        model.unfreeze_all()
        criterion = MultiTaskLoss(
            w_auth=tc.loss_weight_auth,
            w_tumor=tc.loss_weight_tumor,
        )
        ckpt_name = "stage3_joint_best.pt"

    optimizer  = build_optimizer(model, lr, tc)
    scheduler  = build_scheduler(optimizer, tc, epochs, len(train_loader))
    scaler = GradScaler("cuda", enabled=cfg.amp)

    best_val_loss  = float("inf")
    patience_count = 0
    ckpt_path      = os.path.join(cfg.paths.checkpoint_dir, ckpt_name)

    print(f"\n{'='*60}")
    print(f"  STAGE {stage}  ({epochs} epochs, lr={lr})")
    print(f"{'='*60}")

    for epoch in range(start_epoch, epochs + 1):
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler,
            scaler, device, cfg, epoch, writer, stage
        )
        val_loss = validate(
            model, val_loader, criterion, device, cfg, epoch, writer, stage
        )

        print(
            f"  Epoch {epoch:03d}/{epochs:03d}  "
            f"train_loss={train_metrics['loss_total']:.4f}  "
            f"val_loss={val_loss:.4f}"
        )

        # ── Early stopping & checkpoint ───────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            save_checkpoint(model, ckpt_path,
                            {"epoch": epoch, "val_loss": val_loss, "stage": stage})
        else:
            patience_count += 1
            if patience_count >= tc.early_stop_patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    print(f"  Best val_loss (stage {stage}): {best_val_loss:.4f}")
    return ckpt_path


# ─────────────────────────────────────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────────────────────────────────────

def main(args):
    seed_everything(CFG.seed)
    CFG.paths.make_dirs()

    if args.data:
        CFG.data.data_root = args.data

    device = torch.device(
        "cuda" if (CFG.device == "cuda" and torch.cuda.is_available()) else "cpu"
    )
    print(f"\n  Device : {device}")
    if device.type == "cuda":
        print(f"  GPU    : {torch.cuda.get_device_name(0)}")

    # ── Dataloaders ──────────────────────────────────────────────────────────
    loaders = build_dataloaders(CFG)

    # ── Model ────────────────────────────────────────────────────────────────
    model = MultiTaskMRINet(CFG.model).to(device)
    stats = model.count_parameters()
    print(f"  Params : {stats['total']:,}  ({stats['total_MB']:.1f} MB fp32)\n")

    # ── TensorBoard ──────────────────────────────────────────────────────────
    writer = SummaryWriter(log_dir=CFG.paths.log_dir)

    # ── Resume from existing checkpoint ──────────────────────────────────────
    start_stage = int(args.stage) if args.stage else 1

    if args.resume:
        meta = load_checkpoint(model, args.resume, device)
        print(f"  Resumed from {args.resume}  (epoch={meta.get('epoch')})")
        start_stage = meta.get("stage", 1)

    # ── Run stages ───────────────────────────────────────────────────────────
    for stage_num in range(start_stage, 4):
        ckpt_path = run_stage(stage_num, model, loaders, CFG, device, writer)

        # Load best weights of just-completed stage before moving on
        if os.path.exists(ckpt_path):
            load_checkpoint(model, ckpt_path, device)
            print(f"  Loaded best weights from {ckpt_path}")
        else:
            print(f"  Warning: no checkpoint found at {ckpt_path}, continuing with current weights")

    # # ── Final evaluation on test set ─────────────────────────────────────────
    # print("\n" + "="*60)
    # print("  FINAL EVALUATION (test set)")
    # print("="*60)
    # test_loader = loaders[2]
    # metrics = run_evaluation(model, test_loader, device, CFG,
    #                          save_dir=CFG.paths.results_dir,
    #                          split="test")

    # metrics_path = os.path.join(CFG.paths.results_dir, "test_metrics.json")
    # with open(metrics_path, "w") as f:
    #     # Convert non-serializable items
    #     clean = {}
    #     for k, v in metrics.items():
    #         try:
    #             json.dumps(v)
    #             clean[k] = v
    #         except TypeError:
    #             clean[k] = str(v)
    #     json.dump(clean, f, indent=2)
    # print(f"  Metrics saved → {metrics_path}")

    # # ── ONNX export ──────────────────────────────────────────────────────────
    # if CFG.eval.export_onnx:
    #     # from export_onnx import export_model
    #     export_model(model, CFG, device)

    # writer.close()
    print("\n  Training complete.")


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print(os.getcwd())
    parser = argparse.ArgumentParser(description="Train MultiTaskMRINet")
    parser.add_argument("--data",   type=str, default=None,
                        help="Path to dataset root")
    parser.add_argument("--stage",  type=int, default=1, choices=[1, 2, 3],
                        help="Start from stage N (1=auth, 2=tumor, 3=joint)")
    parser.add_argument("--resume", type=str, default=None,
                        help="Path to checkpoint to resume from")
    args = parser.parse_args([])
    main(args)


d:\Semester 6\ANN\Project\AbuzarWork\LightWeight\Submission_Files\Code_Files

  Device : cpu


d:\Semester 6\ANN\Project\AbuzarWork\LightWeight\Submission_Files\submission\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\REHAN\AppData\Local\Temp\ipykernel_13300\3250622850.py:53: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=dc.aug_noise_var_limit, p=0.4),
C:\Users\REHAN\AppData\Local\Temp\ipykernel_13300\3250622850.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=16, max_width=16,


[Dataset] train=4,224  val=600  test=1,204  total=6,028


C:\Users\REHAN\AppData\Local\Temp\ipykernel_13300\378785340.py:248: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler("cuda", enabled=cfg.amp)


  Params : 1,521,974  (6.1 MB fp32)


  STAGE 1  (20 epochs, lr=0.001)


Stage 1  Epoch 001  Train:   0%|                                  | 0/132 [00:00<?, ?it/s]d:\Semester 6\ANN\Project\AbuzarWork\LightWeight\Submission_Files\submission\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


---
## evaluate.py

In [ ]:
"""
evaluate.py
-----------
Evaluation engine for MultiTaskMRINet.

Computes per-task and joint metrics:
  - Authenticity (real/fake): accuracy, AUC, F1
  - Tumor (4-way):            accuracy, F1-macro, F1-per-class, confusion matrix
  - Joint (5-way):            accuracy, F1-macro, confusion matrix

Saves:
  - JSON metrics file
  - Confusion matrix PNG(s)
  - Grad-CAM visualizations (sample batch)
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, List, Optional

import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from tqdm import tqdm

# from config import (CFG, Config, AUTHENTICITY_CLASSES, TUMOR_CLASSES,
#                     FAKE_TYPE_CLASSES, JOINT_CLASSES)


# ─────────────────────────────────────────────────────────────────────────────
# Prediction collector
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def collect_predictions(model, loader, device: torch.device, cfg: Config):
    """
    Run inference over a DataLoader and collect raw logits + ground truth.

    Returns
    -------
    dict with keys:
      auth_preds, auth_probs, auth_gt
      tumor_preds, tumor_gt
      joint_preds, joint_gt
    """
    model.eval()

    auth_preds_all, auth_probs_all, auth_gt_all = [], [], []
    tumor_preds_all, tumor_gt_all                 = [], []
    joint_preds_all, joint_gt_all                 = [], []

    for imgs, auth_gt, tumor_gt, joint_gt in tqdm(
            loader, desc="  Inference", leave=False, ncols=90):

        imgs = imgs.to(device)

        a_logits, t_logits = model(imgs)

        # ── Authenticity ────────────────────────────────────────────────────
        auth_probs = F.softmax(a_logits, dim=-1)   # (B, 2)
        auth_preds = auth_probs.argmax(-1)

        # ── Joint prediction (cascade logic) ────────────────────────────────
        tumor_probs = F.softmax(t_logits, dim=-1)  # (B, 4)
        tumor_preds = tumor_probs.argmax(-1)        # (B,)

        # joint: real→tumor_pred(0-3), fake→4
        is_fake    = (auth_preds == 1)
        joint_pred = torch.where(is_fake, torch.full_like(tumor_preds, 4), tumor_preds)

        # CPU / numpy
        auth_preds_all.extend(auth_preds.cpu().numpy())
        auth_probs_all.extend(auth_probs.cpu().numpy())
        auth_gt_all.extend(auth_gt.numpy())

        tumor_preds_all.extend(tumor_preds.cpu().numpy())
        tumor_gt_all.extend(tumor_gt.numpy())

        joint_preds_all.extend(joint_pred.cpu().numpy())
        joint_gt_all.extend(joint_gt.numpy())

    return {
        "auth_preds":  np.array(auth_preds_all),
        "auth_probs":  np.array(auth_probs_all),
        "auth_gt":     np.array(auth_gt_all),
        "tumor_preds": np.array(tumor_preds_all),
        "tumor_gt":    np.array(tumor_gt_all),
        "joint_preds": np.array(joint_preds_all),
        "joint_gt":    np.array(joint_gt_all),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Metric calculators
# ─────────────────────────────────────────────────────────────────────────────

def _compute_task_metrics(preds, gt, class_names, probs=None, task_name="task"):
    """Compute accuracy, F1, AUC (optional) for a single task."""
    # Filter out masked samples (-1)
    mask = (gt != -1)
    if mask.sum() == 0:
        return {}

    y_true = gt[mask]
    y_pred = preds[mask]

    acc     = accuracy_score(y_true, y_pred)
    f1_mac  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_per  = f1_score(y_true, y_pred, average=None, zero_division=0).tolist()
    report  = classification_report(y_true, y_pred,
                                    target_names=class_names, zero_division=0)

    metrics = {
        f"{task_name}_accuracy":  round(acc, 4),
        f"{task_name}_f1_macro":  round(f1_mac, 4),
        f"{task_name}_f1_per_class": {
            c: round(f, 4) for c, f in zip(class_names, f1_per)
        },
        f"{task_name}_report":    report,
    }

    if probs is not None and len(np.unique(y_true)) == 2:
        try:
            auc = roc_auc_score(y_true, probs[mask][:, 1])
            metrics[f"{task_name}_auc"] = round(float(auc), 4)
        except Exception:
            pass

    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# Confusion matrix plotter
# ─────────────────────────────────────────────────────────────────────────────

def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: List[str],
    title: str,
    save_path: str,
):
    """Save a styled confusion matrix PNG."""
    fig, ax = plt.subplots(figsize=(max(6, len(class_names)),
                                    max(5, len(class_names))))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.5, ax=ax,
    )
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True",      fontsize=12)
    ax.set_title(title,        fontsize=14, pad=12)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  Confusion matrix saved → {save_path}")




# ─────────────────────────────────────────────────────────────────────────────
# Main evaluation entry point
# ─────────────────────────────────────────────────────────────────────────────

def run_evaluation(
    model,
    loader,
    device: torch.device,
    cfg:    Config,
    save_dir: str = "results",
    split:  str   = "val",
    run_gradcam: bool = False,
) -> Dict:
    """
    Full evaluation: all tasks + joint confusion matrix.

    Returns
    -------
    metrics dict (also saved to save_dir/<split>_metrics.json)
    """
    os.makedirs(save_dir, exist_ok=True)
    results = collect_predictions(model, loader, device, cfg)

    metrics: Dict = {}

    # ── 1. Authenticity ──────────────────────────────────────────────────────
    auth_m = _compute_task_metrics(
        results["auth_preds"], results["auth_gt"],
        AUTHENTICITY_CLASSES, probs=results["auth_probs"],
        task_name="auth"
    )
    metrics.update(auth_m)

    cm_auth = confusion_matrix(results["auth_gt"], results["auth_preds"])
    plot_confusion_matrix(
        cm_auth, AUTHENTICITY_CLASSES,
        f"Authenticity – {split}",
        os.path.join(save_dir, f"{split}_cm_auth.png"),
    )

    # ── 2. Tumor (real images only) ──────────────────────────────────────────
    real_mask = results["auth_gt"] == 0
    if real_mask.any():
        tumor_m = _compute_task_metrics(
            results["tumor_preds"][real_mask],
            results["tumor_gt"][real_mask],
            TUMOR_CLASSES, task_name="tumor"
        )
        metrics.update(tumor_m)

        cm_tumor = confusion_matrix(
            results["tumor_gt"][real_mask],
            results["tumor_preds"][real_mask]
        )
        plot_confusion_matrix(
            cm_tumor, TUMOR_CLASSES,
            f"Tumor Classification – {split}",
            os.path.join(save_dir, f"{split}_cm_tumor.png"),
        )

    # ── 3. Joint 5-way ───────────────────────────────────────────────────────
    joint_m = _compute_task_metrics(
        results["joint_preds"], results["joint_gt"],
        JOINT_CLASSES, task_name="joint"
    )
    metrics.update(joint_m)

    cm_joint = confusion_matrix(results["joint_gt"], results["joint_preds"],
                                labels=list(range(len(JOINT_CLASSES))))
    plot_confusion_matrix(
        cm_joint, JOINT_CLASSES,
        f"Joint 5-way – {split}",
        os.path.join(save_dir, f"{split}_cm_joint.png"),
    )

    # ── 4. Print summary ─────────────────────────────────────────────────────
    print(f"\n  ── {split.upper()} METRICS ──────────────────────────────────")
    for k, v in metrics.items():
        if "report" in k:
            print(f"\n  [{k}]\n{v}")
        elif isinstance(v, dict):
            print(f"  {k}:")
            for ck, cv in v.items():
                print(f"      {ck}: {cv}")
        else:
            print(f"  {k}: {v}")

    # ── 5. Optional Grad-CAM ─────────────────────────────────────────────────
    if run_gradcam:
        generate_gradcam(model, loader, device, cfg, save_dir)

    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# Stand-alone evaluation script
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import argparse
    # from model import MultiTaskMRINet
    # from dataset import build_dataloaders

    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True, help="Path to .pt checkpoint")
    parser.add_argument("--data",       default=None,  help="Data root override")
    parser.add_argument("--split",      default="test", choices=["val", "test"])
    parser.add_argument("--gradcam",    action="store_true")
    args = parser.parse_args([
        "--checkpoint", "checkpoints/stage3_joint_best.pt",
        "--split", "test",
        # "--gradcam",   # uncomment if you want Grad-CAM
        # "--data", "/kaggle/working/Ready-Dataset",  # uncomment to override data root
    ])
    if args.data:
        CFG.data.data_root = args.data

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _, val_loader, test_loader = build_dataloaders(CFG)
    loader = test_loader if args.split == "test" else val_loader

    model = MultiTaskMRINet(CFG.model).to(device)
    ckpt  = torch.load(args.checkpoint, map_location=device)
    model.load_state_dict(ckpt["model_state"])

    run_evaluation(model, loader, device, CFG,
                   save_dir=CFG.paths.results_dir,
                   split=args.split,
                   run_gradcam=args.gradcam)

    # # ── Final evaluation on test set ─────────────────────────────────────────
    # print("\n" + "="*60)
    # print("  FINAL EVALUATION (test set)")
    # print("="*60)
    # test_loader = loaders[2]
    # metrics = run_evaluation(model, test_loader, device, CFG,
    #                          save_dir=CFG.paths.results_dir,
    #                          split="test")

    # metrics_path = os.path.join(CFG.paths.results_dir, "test_metrics.json")
    # with open(metrics_path, "w") as f:
    #     # Convert non-serializable items
    #     clean = {}
    #     for k, v in metrics.items():
    #         try:
    #             json.dumps(v)
    #             clean[k] = v
    #         except TypeError:
    #             clean[k] = str(v)
    #     json.dump(clean, f, indent=2)
    # print(f"  Metrics saved → {metrics_path}")

    # # ── ONNX export ──────────────────────────────────────────────────────────
    # if CFG.eval.export_onnx:
    #     # from export_onnx import export_model
    #     export_model(model, CFG, device)

    # writer.close()


/tmp/ipykernel_57/3250622850.py:53: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=dc.aug_noise_var_limit, p=0.4),
/tmp/ipykernel_57/3250622850.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=16, max_width=16,


[Dataset] train=4,224  val=600  test=1,204  total=6,028


  Confusion matrix saved → results/test_cm_auth.png
  Confusion matrix saved → results/test_cm_tumor.png
  Confusion matrix saved → results/test_cm_joint.png

  ── TEST METRICS ──────────────────────────────────
  auth_accuracy: 0.9884
  auth_f1_macro: 0.9814
  auth_f1_per_class:
      real: 0.9928
      fake: 0.97

  [auth_report]
              precision    recall  f1-score   support

        real       0.99      1.00      0.99       964
        fake       1.00      0.94      0.97       240

    accuracy                           0.99      1204
   macro avg       0.99      0.97      0.98      1204
weighted avg       0.99      0.99      0.99      1204

  auth_auc: 0.9996
  tumor_accuracy: 0.9564
  tumor_f1_macro: 0.9564
  tumor_f1_per_class:
      no_tumor: 0.9876
      glioma: 0.9365
      meningioma: 0.924
      pituitary: 0.9774

  [tumor_report]
              precision    recall  f1-score   support

    no_tumor       0.98      0.99      0.99       241
      glioma       0.99      

---
## export_onnx.py

In [8]:
"""
export_onnx.py
--------------
Exports the trained MultiTaskMRINet to ONNX and applies
post-training INT8 quantization via ONNX Runtime.

Usage
-----
    python export_onnx.py --checkpoint checkpoints/stage3_joint_best.pt

Output
------
    export/
        multitask_mri.onnx              # fp32 ONNX
        multitask_mri_int8.onnx         # INT8 quantized
"""

from __future__ import annotations

import argparse
import os
from pathlib import Path

import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np

# from config import CFG, Config


# ─────────────────────────────────────────────────────────────────────────────
# ONNX-friendly wrapper (three outputs concatenated in a dict)
# ─────────────────────────────────────────────────────────────────────────────

class ONNXWrapper(nn.Module):
    """
    Wraps MultiTaskMRINet to emit three named outputs for ONNX.
    Dynamic axes allow variable batch sizes.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        auth, tumor = self.model(x)
        return auth, tumor


# ─────────────────────────────────────────────────────────────────────────────
# Export function
# ─────────────────────────────────────────────────────────────────────────────

def export_model(model, cfg: Config, device: torch.device):
    """Export trained model to ONNX + INT8 quantized ONNX."""
    os.makedirs(cfg.paths.onnx_dir, exist_ok=True)
    onnx_path     = os.path.join(cfg.paths.onnx_dir, "multitask_mri.onnx")
    onnx_int8_path = os.path.join(cfg.paths.onnx_dir, "multitask_mri_int8.onnx")

    model.eval().to(device)
    wrapper = ONNXWrapper(model)

    dummy_input = torch.randn(
        1, 3, cfg.data.image_size, cfg.data.image_size,
        device=device
    )

    print("\n  Exporting to ONNX …")
    torch.onnx.export(
        wrapper,
        dummy_input,
        onnx_path,
        opset_version=cfg.eval.onnx_opset,
        input_names=["input"],
        output_names=["auth_logits", "tumor_logits"],
        dynamic_axes={
            "input":        {0: "batch_size"},
            "auth_logits":  {0: "batch_size"},
            "tumor_logits": {0: "batch_size"},
        },
        export_params=True,
        do_constant_folding=True,
    )

    # ── Verify ONNX graph ────────────────────────────────────────────────────
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f"  ✓ ONNX model verified  → {onnx_path}")

    # ── ONNX Runtime smoke test ──────────────────────────────────────────────
    sess = ort.InferenceSession(onnx_path,
                                providers=["CPUExecutionProvider"])
    dummy_np = dummy_input.cpu().numpy()
    outputs  = sess.run(None, {"input": dummy_np})
    print(f"  ✓ ORT outputs: {[o.shape for o in outputs]}")

    # ── INT8 post-training quantization ──────────────────────────────────────
    if cfg.eval.quantize_int8:
        try:
            from onnxruntime.quantization import quantize_dynamic, QuantType
            quantize_dynamic(
                onnx_path,
                onnx_int8_path,
                weight_type=QuantType.QInt8,
            )
            size_fp32 = Path(onnx_path).stat().st_size / 1e6
            size_int8 = Path(onnx_int8_path).stat().st_size / 1e6
            print(f"  ✓ INT8 model saved    → {onnx_int8_path}")
            print(f"    fp32: {size_fp32:.1f} MB   int8: {size_int8:.1f} MB  "
                  f"({size_int8/size_fp32*100:.0f}%)")
        except Exception as e:
            print(f"  [WARN] INT8 quantization failed: {e}")

    # ── Latency benchmark ────────────────────────────────────────────────────
    _benchmark_ort(onnx_path, dummy_np, label="fp32")
    if cfg.eval.quantize_int8 and Path(onnx_int8_path).exists():
        _benchmark_ort(onnx_int8_path, dummy_np, label="int8")


def _benchmark_ort(onnx_path: str, dummy: np.ndarray,
                   n_runs: int = 100, label: str = ""):
    import time
    sess = ort.InferenceSession(onnx_path,
                                providers=["CPUExecutionProvider"])
    # Warm-up
    for _ in range(10):
        sess.run(None, {"input": dummy})
    t0 = time.perf_counter()
    for _ in range(n_runs):
        sess.run(None, {"input": dummy})
    elapsed = (time.perf_counter() - t0) / n_runs * 1000
    print(f"  ORT latency [{label}] (batch=1, CPU): {elapsed:.2f} ms/image")


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # from model import MultiTaskMRINet

    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", required=True,
                        help="Path to trained .pt checkpoint")
    args = parser.parse_args([
            "--checkpoint", "checkpoints/stage3_joint_best.pt",
            "--output",     "export/multitask_mri.onnx",
        ])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = MultiTaskMRINet(CFG.model).to(device)

    ckpt = torch.load(args.checkpoint, map_location=device)
    model.load_state_dict(ckpt["model_state"])

    export_model(model, CFG, device)


ModuleNotFoundError: No module named 'onnxruntime'

---
## kfold_train.py

In [ ]:
# """
# kfold_train.py
# --------------
# K-Fold cross-validation wrapper around the staged trainer.

# Usage
# -----
#     python kfold_train.py --data data/ --folds 5

# Each fold:
#   1. Splits samples into train / val (test held out globally)
#   2. Runs all 3 training stages
#   3. Evaluates on fold-val and global test set
#   4. Aggregates and reports mean ± std metrics
# """

# from __future__ import annotations

# import argparse
# import json
# import os
# import random
# from collections import defaultdict
# from typing import Dict, List

# import numpy as np
# import torch
# from sklearn.model_selection import StratifiedKFold

# # from config import CFG
# # from dataset import (MRIMultiTaskDataset, build_train_transforms,
# #                      build_val_transforms, make_weighted_sampler, scan_dataset,
# #                      stratified_patient_split)
# # from model import MultiTaskMRINet
# # from train import run_stage, load_checkpoint, seed_everything
# # from evaluate import run_evaluation
# from torch.utils.data import DataLoader
# from torch.utils.tensorboard import SummaryWriter


# def kfold_train(cfg=CFG, n_folds: int = 5):
#     seed_everything(cfg.seed)
#     cfg.paths.make_dirs()

#     if not cfg.data.data_root or not os.path.isdir(cfg.data.data_root):
#         raise FileNotFoundError(f"Data root not found: {cfg.data.data_root}")

#     device = torch.device(
#         "cuda" if (cfg.device == "cuda" and torch.cuda.is_available()) else "cpu"
#     )
#     print(f"\n  K-Fold CV  ({n_folds} folds)  device={device}\n")

#     # ── Load & split off global test set ──────────────────────────────────────
#     all_samples = scan_dataset(cfg.data.data_root)
#     train_val_s, _, test_s = stratified_patient_split(
#         all_samples,
#         val_frac=0.0,
#         test_frac=cfg.data.test_split,
#         seed=cfg.seed,
#     )
#     test_ds = MRIMultiTaskDataset(test_s, build_val_transforms(cfg))
#     test_loader = DataLoader(
#         test_ds, batch_size=cfg.training.stage1_batch * 2,
#         shuffle=False, num_workers=cfg.training.num_workers,
#     )

#     # ── Stratified K-Fold on joint labels ─────────────────────────────────────
#     labels = np.array([s["joint"] for s in train_val_s])
#     skf    = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=cfg.seed)

#     fold_metrics: Dict[str, List[float]] = defaultdict(list)

#     for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_val_s, labels)):
#         print(f"\n{'='*60}")
#         print(f"  FOLD {fold_idx + 1} / {n_folds}")
#         print(f"{'='*60}")

#         fold_train = [train_val_s[i] for i in train_idx]
#         fold_val   = [train_val_s[i] for i in val_idx]

#         sampler = make_weighted_sampler(fold_train)
#         train_loader = DataLoader(
#             MRIMultiTaskDataset(fold_train, build_train_transforms(cfg)),
#             batch_size=cfg.training.stage1_batch, sampler=sampler,
#             num_workers=cfg.training.num_workers, pin_memory=cfg.training.pin_memory,
#             drop_last=True,
#         )
#         val_loader = DataLoader(
#             MRIMultiTaskDataset(fold_val, build_val_transforms(cfg)),
#             batch_size=cfg.training.stage1_batch * 2, shuffle=False,
#             num_workers=cfg.training.num_workers,
#         )

#         model  = MultiTaskMRINet(cfg.model).to(device)
#         writer = SummaryWriter(
#             log_dir=os.path.join(cfg.paths.log_dir, f"fold{fold_idx+1}")
#         )
#         loaders_tuple = (train_loader, val_loader, test_loader)

#         for stage in [1, 2, 3]:
#             ckpt_path = run_stage(stage, model, loaders_tuple, cfg, device, writer)
#             if os.path.exists(ckpt_path):
#                 print(f"  Loaded best weights from {ckpt_path}")
#             else:
#                 print(f"  Warning: no checkpoint found at {ckpt_path}, continuing with current weights")

#         # ── Fold-level evaluation ──────────────────────────────────────────────
#         fold_dir = os.path.join(cfg.paths.results_dir, f"fold{fold_idx+1}")
#         metrics  = run_evaluation(
#             model, val_loader, device, cfg,
#             save_dir=fold_dir, split="val"
#         )

#         for k, v in metrics.items():
#             if isinstance(v, (int, float)):
#                 fold_metrics[k].append(float(v))

#         writer.close()

#     # ── Aggregate ─────────────────────────────────────────────────────────────
#     print(f"\n{'='*60}")
#     print(f"  K-FOLD SUMMARY  ({n_folds} folds)")
#     print(f"{'='*60}")
#     agg = {}
#     for k, vals in fold_metrics.items():
#         mean_v = float(np.mean(vals))
#         std_v  = float(np.std(vals))
#         agg[k] = {"mean": round(mean_v, 4), "std": round(std_v, 4)}
#         print(f"  {k:<40} {mean_v:.4f} ± {std_v:.4f}")

#     out_path = os.path.join(cfg.paths.results_dir, "kfold_summary.json")
#     with open(out_path, "w") as f:
#         json.dump(agg, f, indent=2)
#     print(f"\n  Summary saved → {out_path}")
#     return agg


# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--data",  default=None, help="Data root override")
#     parser.add_argument("--folds", type=int, default=CFG.data.k_folds)
#     args = parser.parse_args([
#         "--data",  "/kaggle/working/Ready-Dataset",
#         "--folds", "5",
#     ])

#     if args.data:
#         CFG.data.data_root = args.data
#     kfold_train(CFG, n_folds=args.folds)


---
## inference.py

In [ ]:
"""
inference.py
------------
Production-ready inference script for MultiTaskMRINet.

Supports:
  - Single image inference
  - Batch folder inference (writes CSV report)
  - ONNX Runtime inference (fp32 or int8)

Usage
-----
    # PyTorch inference on single image
    python inference.py --checkpoint checkpoints/stage3_joint_best.pt \
                        --image path/to/mri.png

    # Folder batch inference
    python inference.py --checkpoint checkpoints/stage3_joint_best.pt \
                        --folder path/to/images/ \
                        --output results/predictions.csv

    # ONNX inference
    python inference.py --onnx export/multitask_mri_int8.onnx \
                        --image path/to/mri.png
"""

from __future__ import annotations

import argparse
import csv
import os
import time
from pathlib import Path
from typing import Dict, Optional

import cv2
import numpy as np
import torch
import torch.nn.functional as F

# from config import (CFG, AUTHENTICITY_CLASSES, TUMOR_CLASSES,
#                     FAKE_TYPE_CLASSES, JOINT_CLASSES)


# ─────────────────────────────────────────────────────────────────────────────
# Preprocessing (mirrors validation transform, no albumentations dependency)
# ─────────────────────────────────────────────────────────────────────────────

def preprocess_image(
    img_path: str,
    image_size: int = 224,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
) -> np.ndarray:
    """Load and preprocess a single MRI image → (1, 3, H, W) float32 numpy."""
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot load image: {img_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32)

    # Per-image intensity normalization (MRI-specific)
    m, s = img.mean(), img.std() + 1e-6
    img  = ((img - m) / s * 50 + 128).clip(0, 255)

    # Resize
    img = cv2.resize(img, (image_size, image_size)).astype(np.float32) / 255.0

    # ImageNet normalize
    mean_arr = np.array(mean, dtype=np.float32)
    std_arr  = np.array(std,  dtype=np.float32)
    img = (img - mean_arr) / std_arr

    # (H, W, C) → (1, C, H, W)
    img = np.transpose(img, (2, 0, 1))[np.newaxis, :]
    return img


# ─────────────────────────────────────────────────────────────────────────────
# PyTorch predictor
# ─────────────────────────────────────────────────────────────────────────────

class TorchPredictor:
    def __init__(self, checkpoint_path: str, device: Optional[str] = None):
        # from model import MultiTaskMRINet
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.model = MultiTaskMRINet(CFG.model).to(self.device)
        ckpt = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()
        print(f"  Loaded PyTorch model from {checkpoint_path}")

    @torch.inference_mode()
    def predict(self, img_path: str) -> Dict:
        arr = preprocess_image(img_path, CFG.data.image_size,
                               CFG.data.mean, CFG.data.std)
        tensor = torch.from_numpy(arr).to(self.device)
        return self.model.predict(tensor)


# ─────────────────────────────────────────────────────────────────────────────
# ONNX Runtime predictor
# ─────────────────────────────────────────────────────────────────────────────

class ONNXPredictor:
    def __init__(self, onnx_path: str):
        import onnxruntime as ort
        providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                     if torch.cuda.is_available() else ["CPUExecutionProvider"])
        self.sess = ort.InferenceSession(onnx_path, providers=providers)
        self.input_name = self.sess.get_inputs()[0].name
        print(f"  Loaded ONNX model from {onnx_path}")

    def predict(self, img_path: str) -> Dict:
        arr = preprocess_image(img_path, CFG.data.image_size,
                               CFG.data.mean, CFG.data.std)
        outputs = self.sess.run(None, {self.input_name: arr})
        auth_logits, tumor_logits = outputs

        auth_probs  = _softmax(auth_logits[0])
        tumor_probs = _softmax(tumor_logits[0])

        auth_pred = int(np.argmax(auth_probs))
        auth_prob = float(np.max(auth_probs))

        if auth_pred == 0:  # real
            tumor_pred = int(np.argmax(tumor_probs))
            return {
                "auth_pred":   auth_pred,
                "auth_label":  AUTHENTICITY_CLASSES[auth_pred],
                "auth_prob":   auth_prob,
                "tumor_pred":  tumor_pred,
                "tumor_label": TUMOR_CLASSES[tumor_pred],
                "tumor_probs": tumor_probs.tolist(),
                "joint_label": JOINT_CLASSES[tumor_pred],
            }
        else:                # fake
            return {
                "auth_pred":   auth_pred,
                "auth_label":  AUTHENTICITY_CLASSES[auth_pred],
                "auth_prob":   auth_prob,
                "tumor_pred":  None,
                "tumor_label": None,
                "tumor_probs": None,
                "joint_label": JOINT_CLASSES[4],
            }


def _softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x))
    return e / e.sum()


# ─────────────────────────────────────────────────────────────────────────────
# Pretty-print result
# ─────────────────────────────────────────────────────────────────────────────

def print_result(img_path: str, result: Dict, elapsed_ms: float):
    auth_lbl = AUTHENTICITY_CLASSES[result["auth_pred"]]
    print(f"\n  Image   : {img_path}")
    print(f"  Auth    : {auth_lbl}  ({result['auth_prob']*100:.1f}%)")
    if result["tumor_label"]:
        probs = {TUMOR_CLASSES[i]: round(result["tumor_probs"][i]*100, 1)
                 for i in range(len(TUMOR_CLASSES))}
        print(f"  Tumor   : {result['tumor_label']}  {probs}")

    print(f"  Joint   : {result['joint_label']}")
    print(f"  Latency : {elapsed_ms:.1f} ms")


# ─────────────────────────────────────────────────────────────────────────────
# Batch folder inference
# ─────────────────────────────────────────────────────────────────────────────

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def batch_infer(predictor, folder: str, output_csv: str):
    """Run inference on all images in `folder` and write CSV."""
    paths = [
        str(p) for p in Path(folder).rglob("*")
        if p.suffix.lower() in VALID_EXTS
    ]
    if not paths:
        print(f"  No images found in {folder}")
        return

    rows = []
    for p in paths:
        try:
            t0 = time.perf_counter()
            r  = predictor.predict(p)
            ms = (time.perf_counter() - t0) * 1000
            rows.append({
                "path":           p,
                "auth_label":  AUTHENTICITY_CLASSES[r["auth_pred"]],
                "auth_prob":   round(r["auth_prob"], 4),
                "tumor_label": r.get("tumor_label") or "",
                "joint_label": r["joint_label"],
                "latency_ms":     round(ms, 2),
            })
        except Exception as e:
            rows.append({"path": p, "error": str(e)})

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        if rows:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
    print(f"  Batch predictions saved → {output_csv}  ({len(rows)} images)")


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="MultiTaskMRINet Inference")
    src = parser.add_mutually_exclusive_group(required=True)
    src.add_argument("--image",  help="Single image path")
    src.add_argument("--folder", help="Folder for batch inference")

    backend = parser.add_mutually_exclusive_group(required=True)
    backend.add_argument("--checkpoint", help="PyTorch .pt checkpoint")
    backend.add_argument("--onnx",       help="ONNX model file (.onnx)")

    parser.add_argument("--output", default="predictions.csv",
                        help="CSV output path for batch inference")
    # Single Image inference
#     args = parser.parse_args([   # ✅
#     "--checkpoint", "checkpoints/stage3_joint_best.pt",
#     "--output",     "export/multitask_mri.onnx",
#     # "--opset",    "17",       # uncomment to override opset version
#     # "--int8",                 # uncomment to enable INT8 quantization
# ])


    # Batch folder inference
    args = parser.parse_args([
        "--checkpoint", "checkpoints/stage3_joint_best.pt",
        "--folder",     "/kaggle/working/Ready-Dataset/real/glioma/",
        "--output",     "predictions.csv",
    ])

    # ── Build predictor ───────────────────────────────────────────────────────
    if args.checkpoint:
        predictor = TorchPredictor(args.checkpoint)
    else:
        predictor = ONNXPredictor(args.onnx)

    # ── Run ───────────────────────────────────────────────────────────────────
    if args.image:
        t0 = time.perf_counter()
        result = predictor.predict(args.image)
        ms     = (time.perf_counter() - t0) * 1000
        print_result(args.image, result, ms)
    else:
        batch_infer(predictor, args.folder, args.output)
